# Session 2 · Part 2 — Create every DGAT model component

**Goal:** understand and instantiate the four modules that are trained jointly. This lesson follows the
official `Model/dgat.py`: separate RNA and protein graph-attention encoders, an RNA decoder, and a
branched protein decoder. The lightweight path prints the architecture and creates a figure; an optional
cell instantiates the real PyTorch modules when the official DGAT environment and repository are present.


In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
for candidate in (current, *current.parents):
    if (candidate / "src" / "dgat_tutorial").is_dir():
        tutorial_root = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter inside the hands-on_tutorial directory.")

sys.path.insert(0, str(tutorial_root / "src"))

from dgat_tutorial.checkpoints import tutorial_paths, write_checkpoint

paths = tutorial_paths(tutorial_root)
print(f"Tutorial root: {paths.root}")


## 1. Set dimensions from the processed data


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

from dgat_tutorial.data import load_tutorial_data
from dgat_tutorial.teaching import official_dgat_component_table

dataset = load_tutorial_data(paths.raw_data)
numeric_rna = dataset.transcripts.select_dtypes(include=[np.number])
numeric_protein = dataset.proteins.select_dtypes(include=[np.number])
common_genes = list(numeric_rna.columns)
common_proteins = list(numeric_protein.columns)
HIDDEN_DIM = 512
component_table = official_dgat_component_table(len(common_genes), common_proteins, HIDDEN_DIM)
component_table


## 2. Read the architecture from left to right

1. Each encoder performs three graph-attention stages with skip projections and LayerNorm.
2. After the first GAT stage, 16 feature-attention heads learn channel gates and their mean reweights the
   2048-dimensional representation.
3. Both encoders end in the same latent dimension so paired RNA and protein spots can be aligned.
4. The protein decoder shares two layers, then uses one output branch per protein. This lets proteins
   share signal while retaining protein-specific prediction heads.


### Figure 6 — DGAT architecture and inference path


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(0, 12); ax.set_ylim(0, 6); ax.axis("off")

def box(x, y, w, h, label, color):
    patch = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.05", facecolor=color, edgecolor="#333333")
    ax.add_patch(patch); ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=9)
def arrow(x1, y1, x2, y2, style="-"):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="->", mutation_scale=12,
                                 linewidth=1.2, linestyle=style, color="#333333"))

box(0.3, 4.1, 1.6, 1.0, "RNA graph\nX_RNA, E_RNA", "#d9eaf7")
box(2.5, 4.1, 2.0, 1.0, "RNA GAT encoder", "#96c5e8")
box(5.2, 4.1, 1.6, 1.0, "z_RNA", "#e4d7f4")
box(0.3, 1.0, 1.6, 1.0, "Protein graph\nX_P, E_P", "#fde2cd")
box(2.5, 1.0, 2.0, 1.0, "Protein GAT encoder", "#f5b97f")
box(5.2, 1.0, 1.6, 1.0, "z_protein", "#e4d7f4")
box(8.0, 4.1, 1.6, 1.0, "RNA decoder", "#d7ecd9")
box(8.0, 1.0, 1.6, 1.0, "Protein decoder", "#d7ecd9")
box(10.2, 4.1, 1.5, 1.0, "RNA output", "#eeeeee")
box(10.2, 1.0, 1.5, 1.0, "Protein output", "#eeeeee")
for start, end in [((1.9,4.6),(2.5,4.6)),((4.5,4.6),(5.2,4.6)),((1.9,1.5),(2.5,1.5)),
                   ((4.5,1.5),(5.2,1.5)),((6.8,4.6),(8.0,4.6)),((9.6,4.6),(10.2,4.6)),
                   ((6.8,1.5),(8.0,1.5)),((9.6,1.5),(10.2,1.5))]: arrow(*start,*end)
arrow(6.8, 4.4, 8.0, 1.8, "--")
arrow(6.8, 1.7, 8.0, 4.3, "--")
ax.text(7.25, 3.0, "cross-modal\nprediction", ha="center", va="center", fontsize=9)
ax.text(6.0, 3.0, "latent alignment", ha="center", va="center", fontsize=9, rotation=90)
ax.plot([6.0,6.0],[2.0,4.1], color="#6b4c9a", linestyle=":", linewidth=2)
architecture_path = paths.figures / "session02_dgat_architecture.png"
fig.savefig(architecture_path, dpi=180, bbox_inches="tight")
plt.show()


## 3. Instantiate the official modules (optional official environment)


In [ ]:
# This is the exact constructor pattern used by the upstream training workflow.
# It executes only when external/DGAT and torch/torch_geometric are available.
dgat_repo = paths.root / "external" / "DGAT"
try:
    import torch
    sys.path.insert(0, str(dgat_repo))
    from Model.dgat import GATEncoder, Decoder_Protein, Decoder_mRNA
    OFFICIAL_READY = dgat_repo.is_dir()
except (ImportError, OSError):
    OFFICIAL_READY = False

if OFFICIAL_READY:
    encoder_rna = GATEncoder(in_channels=len(common_genes), hidden_dim=HIDDEN_DIM, dropout=0.4)
    decoder_rna = Decoder_mRNA(HIDDEN_DIM, len(common_genes), dropout=0)
    encoder_protein = GATEncoder(in_channels=len(common_proteins), hidden_dim=HIDDEN_DIM, dropout=0.4)
    decoder_protein = Decoder_Protein(HIDDEN_DIM, common_proteins, dropout=0)
    modules = {"RNA encoder": encoder_rna, "RNA decoder": decoder_rna,
               "protein encoder": encoder_protein, "protein decoder": decoder_protein}
    for name, module in modules.items():
        print(f"{name:18s}: {sum(p.numel() for p in module.parameters()):,} parameters")
else:
    print("Architecture lesson complete. To instantiate modules, use environment-dgat-cpu.yml and clone DGAT to external/DGAT.")


In [ ]:
component_path = paths.results / "session02_dgat_components.csv"
component_table.to_csv(component_path, index=False)
manifest = write_checkpoint(
    "2.2", [component_path, architecture_path],
    summary={"modules": len(component_table), "official_modules_instantiated": OFFICIAL_READY}, start=paths.root,
)
print(f"Checkpoint written: {manifest}")


## Check

Point to the exact inference route in the figure: **RNA graph → RNA encoder → z_RNA → protein
decoder**. The protein encoder and RNA decoder are training-time partners that make the shared latent
space learnable; they are not needed to impute protein on an RNA-only sample.
